<a href="https://colab.research.google.com/github/Robin-FIME/InteligenciaArtificial_y_RedesNeuronales_UANL_FIME/blob/main/ACTIVIDADES/PIA_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Producto Integrador de Aprendizaje: Sistema de aprendizaje automático con árbol de decisiones mediante redes neuronales formada por pesos**
-**Grupo:** 004 -**Horario de clase:** Jueves N1 a N3
* **2066292**	Gabriel Alejandro Almanza Mata	**IMC**

* **2100051**	Alfredo René Robles Guzman	**IMC**

* **2177682**	Luis Mauricio Torres Mayorga **IMC**

* **2111623** Valeria Monserrat Rincón Vásquez **IMC**

* **1930458** Juan Carlos Torres Gonzales **IMC**


In [12]:
import math
import json
import os
import time
import base64
import numpy as np
import cv2

try:
    from google.colab import output, files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# =====================================================
# CONFIGURACION AI-THINKER ESP32-CAM (REFERENCIA)
# =====================================================

PWDN_GPIO_NUM     = 32
RESET_GPIO_NUM    = -1
XCLK_GPIO_NUM     = 0
SIOD_GPIO_NUM     = 26
SIOC_GPIO_NUM     = 27
Y9_GPIO_NUM       = 35
Y8_GPIO_NUM       = 34
Y7_GPIO_NUM       = 39
Y6_GPIO_NUM       = 36
Y5_GPIO_NUM       = 21
Y4_GPIO_NUM       = 19
Y3_GPIO_NUM       = 18
Y2_GPIO_NUM       = 5
VSYNC_GPIO_NUM    = 25
HREF_GPIO_NUM     = 23
PCLK_GPIO_NUM     = 22

# PIN DE SALIDA UART
PIN_SALIDA_COLOR  = 13

# UART2: 9600 BAUDIOS
ultimoColorEnviado = '\0'

ARCHIVO_MEMORIA = "ia_color.json"

# =====================================================
# RED NEURONAL: TRES NEURONAS
# 0 = ROJO | 1 = VERDE | 2 = AZUL
# =====================================================

W = [[0.0, 0.0, 0.0],
     [0.0, 0.0, 0.0],
     [0.0, 0.0, 0.0]]
bias = [0.0, 0.0, 0.0]

nodoUmbralLuz   = 0.16
nodoUmbralCroma = 0.10
nodoConfianza   = 0.55

MARGEN_NEURONAL = 0.20

ALPHA_SUPERVISADO = 0.35
ALPHA_AUTONOMO    = 0.015

ultimo_rn = 0.0
ultimo_gn = 0.0
ultimo_bn = 0.0

ultima_luz   = 0.0
ultimo_croma = 0.0

hayLectura = False

In [13]:
# =====================================================
# TRANSMISION DEL COLOR POR UART2
# =====================================================

def enviarColor(color):
    global ultimoColorEnviado

    if color != '\0':
        if color != ultimoColorEnviado:
            # Equivalente a Serial2.write(color)
            print(f"[UART] Caracter enviado: {color} por GPIO {PIN_SALIDA_COLOR}")
            ultimoColorEnviado = color
    else:
        ultimoColorEnviado = '\0'

# =====================================================
# FUNCION SIGMOIDE
# =====================================================

def sigmoide(z):
    return 1.0 / (1.0 + math.exp(-z))

# =====================================================
# RESTAURAR PESOS DE FABRICA
# =====================================================

def restaurarPesosFabrica():
    global W, bias, nodoUmbralLuz, nodoUmbralCroma

    W[0][0] = 5.0
    W[0][1] = -2.8
    W[0][2] = -2.8
    bias[0] = -0.4

    W[1][0] = -2.8
    W[1][1] = 5.0
    W[1][2] = -2.8
    bias[1] = -0.4

    W[2][0] = -2.8
    W[2][1] = -2.8
    W[2][2] = 5.0
    bias[2] = -0.4

    nodoUmbralLuz = 0.16
    nodoUmbralCroma = 0.10

In [14]:
# =====================================================
# MEMORIA PERMANENTE
# =====================================================

def guardarConocimiento():
    datos = {
        "pesos": W,
        "bias": bias,
        "u_luz": nodoUmbralLuz,
        "u_croma": nodoUmbralCroma,
        "entrenado": True
    }

    with open(ARCHIVO_MEMORIA, "w") as f:
        json.dump(datos, f)

    print("[MEMORIA] Conocimiento guardado en Flash.")

def cargarConocimiento():
    global W, bias, nodoUmbralLuz, nodoUmbralCroma

    if os.path.exists(ARCHIVO_MEMORIA):
        try:
            with open(ARCHIVO_MEMORIA, "r") as f:
                datos = json.load(f)

            entrenado = datos.get("entrenado", False)

            if entrenado:
                pesos_guardados = datos.get("pesos", [])
                bias_guardados = datos.get("bias", [])

                if len(pesos_guardados) == 3 and all(len(fila) == 3 for fila in pesos_guardados) and len(bias_guardados) == 3:
                    W = [list(fila) for fila in pesos_guardados]
                    bias = list(bias_guardados)
                    nodoUmbralLuz = float(datos.get("u_luz", 0.16))
                    nodoUmbralCroma = float(datos.get("u_croma", 0.10))

                    print("[IA] Conocimiento anterior recuperado.")
                    print("[IA] Pesos neuronales cargados correctamente.")
                else:
                    restaurarPesosFabrica()
                    print("[IA] Memoria incompleta. Se restauran los valores de fabrica.")
            else:
                restaurarPesosFabrica()
                print("[IA] Iniciando con pesos neuronales de fabrica.")
        except Exception:
            restaurarPesosFabrica()
            print("[IA] Memoria incompleta. Se restauran los valores de fabrica.")
    else:
        restaurarPesosFabrica()
        print("[IA] Iniciando con pesos neuronales de fabrica.")

In [15]:
# =====================================================
# CALCULO DE SALIDA DE LAS NEURONAS
# =====================================================

def calcularNeuronas(prob):
    entradas = [
        ultimo_rn,
        ultimo_gn,
        ultimo_bn
    ]

    for i in range(3):
        z = (
            W[i][0] * entradas[0] +
            W[i][1] * entradas[1] +
            W[i][2] * entradas[2] +
            bias[i]
        )
        prob[i] = sigmoide(z)

# =====================================================
# CALCULO DEL ERROR DE APRENDIZAJE
# =====================================================

def calcularErrorMedio(claseObjetivo):
    prob = [0.0, 0.0, 0.0]
    calcularNeuronas(prob)

    errorTotal = 0.0

    for i in range(3):
        deseado = 1.0 if (i == claseObjetivo) else 0.0
        errorTotal += abs(deseado - prob[i])

    return errorTotal / 3.0

# =====================================================
# ENTRENAMIENTO DE LA RED NEURONAL
# =====================================================

def entrenarRedNeuronal(claseObjetivo, tasaAprendizaje):
    global W, bias, nodoUmbralLuz, nodoUmbralCroma

    entradas = [
        ultimo_rn,
        ultimo_gn,
        ultimo_bn
    ]

    for i in range(3):
        z = (
            W[i][0] * entradas[0] +
            W[i][1] * entradas[1] +
            W[i][2] * entradas[2] +
            bias[i]
        )

        prediccion = sigmoide(z)
        deseado = 1.0 if (i == claseObjetivo) else 0.0
        error = deseado - prediccion

        for j in range(3):
            W[i][j] += tasaAprendizaje * error * entradas[j]

        bias[i] += tasaAprendizaje * error

    if ultima_luz < nodoUmbralLuz:
        nodoUmbralLuz = max(0.05, ultima_luz * 0.85)

    if ultimo_croma < nodoUmbralCroma:
        nodoUmbralCroma = max(0.03, ultimo_croma * 0.85)

# =====================================================
# MOSTRAR PESOS NEURONALES
# =====================================================

def mostrarPesos():
    nombres = [
        "ROJO ",
        "VERDE",
        "AZUL "
    ]

    print("\n========== ESTADO DE LA RED NEURONAL ==========")
    print(f"Umbral de luz: {nodoUmbralLuz:.3f}")
    print(f"Umbral de croma: {nodoUmbralCroma:.3f}")

    for i in range(3):
        print(
            f"{nombres[i]} | Wr: {W[i][0]:7.3f} | Wg: {W[i][1]:7.3f} | Wb: {W[i][2]:7.3f} | Bias: {bias[i]:7.3f}"
        )

    print("===============================================\n")

In [16]:
# =====================================================
# ENTRENAMIENTO SUPERVISADO CON EVIDENCIA
# =====================================================

def ejecutarEntrenamiento(claseObjetivo, nombre):
    if not hayLectura:
        print("[ERROR] Espera la primera lectura de la camara.")
        return

    print("\n==============================================")
    print(f"[APRENDIZAJE] Enseñando color: {nombre}")

    print("[1] Datos de entrada:")
    print(f"R normalizado: {ultimo_rn:.4f}")
    print(f"G normalizado: {ultimo_gn:.4f}")
    print(f"B normalizado: {ultimo_bn:.4f}")

    print("\n[2] Pesos ANTES del entrenamiento:")
    mostrarPesos()

    errorAntes = calcularErrorMedio(claseObjetivo)
    print(f"Error medio inicial: {errorAntes:.5f}")

    print("\n[3] Iniciando 8 epocas de aprendizaje...")

    for epoca in range(8):
        entrenarRedNeuronal(claseObjetivo, ALPHA_SUPERVISADO)
        errorActual = calcularErrorMedio(claseObjetivo)
        print(f"Epoca {epoca + 1} | Error medio: {errorActual:.5f}")

    errorFinal = calcularErrorMedio(claseObjetivo)

    print("\n[4] Pesos DESPUES del entrenamiento:")
    mostrarPesos()

    print(f"Error inicial: {errorAntes:.5f}")
    print(f"Error final: {errorFinal:.5f}")

    guardarConocimiento()

    print("[5] Aprendizaje guardado permanentemente.")
    print("==============================================\n")

# =====================================================
# COMANDOS DEL MONITOR SERIAL
# =====================================================

def revisarComandosEntrenamiento(entrada_usuario=""):
    global nodoUmbralCroma, nodoUmbralLuz

    if not entrada_usuario:
        return

    cmd = entrada_usuario.strip()[0].upper()

    if cmd == 'R':
        ejecutarEntrenamiento(0, "ROJO")

    elif cmd == 'V':
        ejecutarEntrenamiento(1, "VERDE")

    elif cmd == 'A':
        ejecutarEntrenamiento(2, "AZUL")

    elif cmd == 'F':
        if not hayLectura:
            return
        nodoUmbralCroma = min(0.45, ultimo_croma + 0.03)
        guardarConocimiento()
        print(f"[ARBOL] Fondo registrado. Nuevo umbral: {nodoUmbralCroma:.3f}")

    elif cmd == 'N':
        if not hayLectura:
            return
        nodoUmbralLuz = min(0.50, ultima_luz + 0.03)
        guardarConocimiento()
        print(f"[ARBOL] Oscuridad registrada. Nuevo umbral: {nodoUmbralLuz:.3f}")

    elif cmd == 'P':
        mostrarPesos()

    elif cmd == 'X':
        restaurarPesosFabrica()
        guardarConocimiento()
        print("[RESET] Pesos y umbrales restaurados de fabrica.")

    else:
        print("[COMANDO] Comando no reconocido.")

In [17]:
# =====================================================
# ADAPTADOR DE CAMARA PARA GOOGLE COLAB (QVGA 320x240)
# =====================================================

def esp_camera_fb_get_colab(modo="webcam"):
    """
    Obtiene un frame RGB de 320x240 (FRAMESIZE_QVGA):
    - modo='webcam': captura desde la webcam del navegador en Colab.
    - modo='imagen': permite subir una imagen desde tu PC.
    - modo='manual': permite escribir directamente valores R, G, B.
    """
    if modo == "manual":
        raw = input("Ingresa RGB separado por comas (ej. 210,35,40): ")
        r, g, b = [int(np.clip(int(x.strip()), 0, 255)) for x in raw.split(",")]
        frame = np.zeros((240, 320, 3), dtype=np.uint8)
        frame[:, :] = [r, g, b]
        return frame

    if modo == "imagen" and EN_COLAB:
        subidos = files.upload()
        if not subidos:
            return None
        nombre_archivo = next(iter(subidos))
        img_bgr = cv2.imread(nombre_archivo)
        if img_bgr is None:
            return None
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        return cv2.resize(img_rgb, (320, 240))

    if modo == "webcam" and EN_COLAB:
        js = """
        async function capturarFrame() {
          const video = document.createElement('video');
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          video.srcObject = stream;
          await video.play();
          await new Promise(resolve => setTimeout(resolve, 400));
          const canvas = document.createElement('canvas');
          canvas.width = 320;
          canvas.height = 240;
          canvas.getContext('2d').drawImage(video, 0, 0, 320, 240);
          stream.getVideoTracks()[0].stop();
          return canvas.toDataURL('image/jpeg', 0.9);
        }
        capturarFrame();
        """
        try:
            data_url = output.eval_js(js)
            header, encoded = data_url.split(",", 1)
            img_bytes = base64.b64decode(encoded)
            np_arr = np.frombuffer(img_bytes, np.uint8)
            img_bgr = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
            return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        except Exception as e:
            print(f"[CAMARA] No se pudo acceder a la webcam ({e}).")
            return None

    return None

# =====================================================
# CONFIGURACION INICIAL
# =====================================================

def setup():
    print("\n==============================================")
    print(" SISTEMA INTELIGENTE DE CLASIFICACION RGB")
    print(" ARBOL DE DECISION + RED NEURONAL")
    print("==============================================")

    print(f"[UART] Salida de colores en GPIO {PIN_SALIDA_COLOR}")
    print("[UART] Velocidad: 9600 baudios.")

    print("[CAMARA] Inicializacion correcta (QVGA 320x240).")

    cargarConocimiento()

    print("\nCOMANDOS DE APRENDIZAJE:")
    print("R = Enseñar ROJO")
    print("V = Enseñar VERDE")
    print("A = Enseñar AZUL")
    print("F = Registrar FONDO")
    print("N = Registrar NEGRO")
    print("P = Consultar PESOS")
    print("X = RESTAURAR valores de fabrica")

    print("\nSistema preparado para clasificar.\n")

In [19]:
# =====================================================
# BUCLE PRINCIPAL
# =====================================================

def loop(modo_captura="webcam"):
    global ultima_luz, ultimo_croma, ultimo_rn, ultimo_gn, ultimo_bn, hayLectura

    comando_pendiente = ""

    while True:
        revisarComandosEntrenamiento(comando_pendiente)
        comando_pendiente = ""

        fb = esp_camera_fb_get_colab(modo=modo_captura)

        if fb is None:
            print("[ERROR] Fallo al capturar imagen.")
            time.sleep(0.5)
            opcion = input("Presiona Enter para reintentar o 'Q' para salir: ").strip().upper()
            if opcion == 'Q':
                break
            continue

        height, width, _ = fb.shape

        # REGION CENTRAL DE 20 x 20 PIXELES
        box_size = 20

        start_x = (width - box_size) // 2
        start_y = (height - box_size) // 2

        sum_r = 0
        sum_g = 0
        sum_b = 0

        total_pixels = box_size * box_size

        for y in range(start_y, start_y + box_size):
            for x in range(start_x, start_x + box_size):
                r, g, b = fb[y, x]
                sum_r += int(r)
                sum_g += int(g)
                sum_b += int(b)

        avg_r = sum_r // total_pixels
        avg_g = sum_g // total_pixels
        avg_b = sum_b // total_pixels

        # ===================================================
        # EXTRACCION Y NORMALIZACION DE CARACTERISTICAS
        # ===================================================

        max_c = max(avg_r, max(avg_g, avg_b)) / 255.0
        min_c = min(avg_r, min(avg_g, avg_b)) / 255.0

        ultima_luz = max_c
        ultimo_croma = max_c - min_c

        suma_total = float(avg_r + avg_g + avg_b) + 1.0

        ultimo_rn = avg_r / suma_total
        ultimo_gn = avg_g / suma_total
        ultimo_bn = avg_b / suma_total

        hayLectura = True

        # ===================================================
        # ARBOL DE DECISIONES
        # ===================================================

        decisionFinal = "INDEFINIDO / FONDO"
        confianzaGanadora = 0.0
        prob = [0.0, 0.0, 0.0]
        margen = 0.0
        neuronaGanadora = 0
        segundaNeurona = -1

        print("\n------------- EVALUACION DEL SISTEMA -------------")
        print(f"RGB: ({avg_r},{avg_g},{avg_b})")

        # NODO 1: ILUMINACION
        print("[ARBOL] NODO 1: Evaluando iluminacion...")
        print(f"Luz: {ultima_luz:.3f} | Umbral: {nodoUmbralLuz:.3f}")

        if ultima_luz < nodoUmbralLuz:
            print("[DECISION] Iluminacion insuficiente.")
            decisionFinal = "OSCURO / NEGRO"
            confianzaGanadora = (1.0 - ultima_luz / nodoUmbralLuz) * 100.0

        # NODO 2: PUREZA CROMATICA
        else:
            print("[DECISION] Iluminacion suficiente.")
            print("[ARBOL] NODO 2: Evaluando pureza cromatica...")
            print(f"Croma: {ultimo_croma:.3f} | Umbral: {nodoUmbralCroma:.3f}")

            if ultimo_croma < nodoUmbralCroma:
                print("[DECISION] Pureza insuficiente.")
                decisionFinal = "INDEFINIDO / FONDO"
                confianzaGanadora = (1.0 - ultimo_croma / nodoUmbralCroma) * 100.0

            # NODO 3: EVALUACION NEURONAL
            else:
                print("[DECISION] Pureza suficiente.")
                print("[ARBOL] NODO 3: Activando red neuronal.")

                calcularNeuronas(prob)

                # BUSCAR LA NEURONA CON MAYOR SALIDA
                for i in range(3):
                    if prob[i] > prob[neuronaGanadora]:
                        neuronaGanadora = i

                # BUSCAR LA SEGUNDA MAYOR SALIDA
                segundaNeurona = -1
                for i in range(3):
                    if i != neuronaGanadora:
                        if segundaNeurona == -1 or prob[i] > prob[segundaNeurona]:
                            segundaNeurona = i

                confianzaGanadora = prob[neuronaGanadora] * 100.0
                margen = prob[neuronaGanadora] - prob[segundaNeurona]

                print(f"Neurona ROJO: {prob[0]:.3f}")
                print(f"Neurona VERDE: {prob[1]:.3f}")
                print(f"Neurona AZUL: {prob[2]:.3f}")
                print(f"Separacion neuronal: {margen:.3f}")

                # VALIDACION FINAL
                if prob[neuronaGanadora] >= nodoConfianza and margen >= MARGEN_NEURONAL:
                    if neuronaGanadora == 0:
                        decisionFinal = "ROJO"
                    elif neuronaGanadora == 1:
                        decisionFinal = "VERDE"
                    else:
                        decisionFinal = "AZUL"

                    print("[RED] Clasificacion aceptada.")

                    # AUTOAPRENDIZAJE
                    if prob[neuronaGanadora] > 0.82:
                        print("[IA] Activando adaptacion autonoma.")
                        errorAntes = calcularErrorMedio(neuronaGanadora)
                        entrenarRedNeuronal(neuronaGanadora, ALPHA_AUTONOMO)
                        errorDespues = calcularErrorMedio(neuronaGanadora)

                        print(f"[IA] Error antes: {errorAntes:.5f} | Despues: {errorDespues:.5f}")
                        guardarConocimiento()
                else:
                    print("[RED] Clasificacion rechazada por confianza o ambiguedad.")
                    decisionFinal = "INDEFINIDO / FONDO"

        # ===================================================
        # TRANSMISION DEL COLOR
        # ===================================================

        codigoColor = '\0'

        if decisionFinal == "ROJO":
            codigoColor = 'R'
        elif decisionFinal == "VERDE":
            codigoColor = 'V'
        elif decisionFinal == "AZUL":
            codigoColor = 'A'

        enviarColor(codigoColor)

        # ===================================================
        # RESULTADO FINAL
        # ===================================================

        print("\n============== RESULTADO FINAL ==============")
        print(f"RGB({avg_r},{avg_g},{avg_b})")
        print(f"Salida neuronal: {confianzaGanadora:.2f}%")
        print(f"COLOR IDENTIFICADO: {decisionFinal}")
        print("=============================================")

        time.sleep(0.3)

        # Interaccion equivalente al Monitor Serial
        entrada = input("\nComando [R/V/A/F/N/P/X], Enter para nueva lectura, o 'Q' para detener: ").strip()
        if entrada.upper() == 'Q':
            print("[SISTEMA] Ejecucion detenida.")
            break
        comando_pendiente = entrada

# =====================================================
# EJECUCION PRINCIPAL
# =====================================================
setup()
# Puedes cambiar modo_captura a: "webcam", "imagen" o "manual"
loop(modo_captura="webcam")


 SISTEMA INTELIGENTE DE CLASIFICACION RGB
 ARBOL DE DECISION + RED NEURONAL
[UART] Salida de colores en GPIO 13
[UART] Velocidad: 9600 baudios.
[CAMARA] Inicializacion correcta (QVGA 320x240).
[IA] Iniciando con pesos neuronales de fabrica.

COMANDOS DE APRENDIZAJE:
R = Enseñar ROJO
V = Enseñar VERDE
A = Enseñar AZUL
F = Registrar FONDO
N = Registrar NEGRO
P = Consultar PESOS
X = RESTAURAR valores de fabrica

Sistema preparado para clasificar.


------------- EVALUACION DEL SISTEMA -------------
RGB: (85,47,41)
[ARBOL] NODO 1: Evaluando iluminacion...
Luz: 0.333 | Umbral: 0.160
[DECISION] Iluminacion suficiente.
[ARBOL] NODO 2: Evaluando pureza cromatica...
Croma: 0.173 | Umbral: 0.100
[DECISION] Pureza suficiente.
[ARBOL] NODO 3: Activando red neuronal.
Neurona ROJO: 0.652
Neurona VERDE: 0.254
Neurona AZUL: 0.207
Separacion neuronal: 0.398
[RED] Clasificacion aceptada.
[UART] Caracter enviado: R por GPIO 13

============== RESULTADO FINAL ==============
RGB(85,47,41)
Salida neuronal: